# 🐝 BEEacon LM
### Plataforma local de agentes RAG especializados

---

Carga tus fuentes de conocimiento, organízalas en **proyectos** y obtén respuestas
fundamentadas en tu contenido. Funciona como un NotebookLM local, con memoria
persistente, múltiples bases de conocimiento y sin límites de privacidad.

**Arquitectura:**
```
Fuentes (PDF / DOCX / PPTX / TXT / MD / URL)
    → chunking  →  Voyage AI embeddings (512 dims)
    → MongoDB Atlas  proj_<id>  +  beaconlm_projects (directorio)
    → Agente RAG (qwen2.5:14b via Ollama)  →  Chat con memoria
```

**Secciones:**
1. 📦 Setup *(ejecutar siempre al iniciar)*
2. 📁 Gestión de proyectos *(crear, listar, activar)*
3. 📂 Carga de documentos *(ejecutar una vez por proyecto)*
4. 🧠 Agente RAG
5. 💬 Chat interactivo
6. 🔍 Diagnóstico *(opcional)*

---
## 📦 Sección 1 — Setup

Ejecuta esta sección completa siempre al iniciar el notebook.
Configura SSL, credenciales, system prompt global y conexión a MongoDB.

In [1]:
import os
import subprocess
import sys

def setup_ssl():
    """
    Configura certificados SSL corporativos automáticamente.
    - macOS:   exporta desde el Keychain del sistema
    - Windows: exporta desde el Certificate Store de Windows
    - Linux:   usa /etc/ssl/certs del sistema
    """
    import certifi
    import tempfile

    certifi_bundle = certifi.where()
    with open(certifi_bundle, 'r') as f:
        bundle_content = f.read()

    if sys.platform == 'darwin':
        system_pem = os.path.expanduser('~/Desktop/system-certs.pem')

        if not os.path.exists(system_pem):
            print('📦 Exportando certificados del sistema macOS...')
            subprocess.run(['security', 'export', '-t', 'certs', '-f', 'pemseq',
                '-k', '/Library/Keychains/System.keychain', '-o', system_pem],
                check=True, capture_output=True)
            login_tmp = tempfile.mktemp(suffix='.pem')
            subprocess.run(['security', 'export', '-t', 'certs', '-f', 'pemseq',
                '-k', os.path.expanduser('~/Library/Keychains/login.keychain-db'),
                '-o', login_tmp], check=True, capture_output=True)
            with open(system_pem, 'a') as f_out:
                with open(login_tmp, 'r') as f_in:
                    f_out.write(f_in.read())
            os.remove(login_tmp)
            print(f'✅ Certificados exportados a {system_pem}')

        kasp = subprocess.run(['security', 'find-certificate', '-c', 'Kaspersky', '-a', '-p'],
            capture_output=True, text=True)
        kasp_cert = kasp.stdout.strip()

        with open(system_pem, 'r') as f:
            system_content = f.read()

        if kasp_cert and kasp_cert[:64] not in bundle_content:
            print('🔐 Agregando certificados corporativos a certifi...')
            with open(certifi_bundle, 'a') as f:
                f.write('\n' + system_content)
            print('✅ Certificados agregados al bundle de certifi')
        else:
            print('✅ Certificados corporativos ya presentes en certifi')

        os.environ['SSL_CERT_FILE']      = system_pem
        os.environ['REQUESTS_CA_BUNDLE'] = system_pem
        os.environ['CURL_CA_BUNDLE']     = system_pem
        print('✅ Variables de entorno SSL configuradas')

    elif sys.platform == 'win32':
        import ssl, base64
        print('📦 Leyendo certificados del Certificate Store de Windows...')
        new_certs = []
        for store_name in ('ROOT', 'CA', 'MY'):
            try:
                for cert_data, encoding, trust in ssl.enum_certificates(store_name):
                    if encoding == 'x509_asn':
                        pem = ('-----BEGIN CERTIFICATE-----\n'
                               + base64.b64encode(cert_data).decode('ascii')
                               + '\n-----END CERTIFICATE-----\n')
                        cert_id = base64.b64encode(cert_data).decode('ascii')[:64]
                        if cert_id not in bundle_content:
                            new_certs.append(pem)
            except Exception:
                continue
        if new_certs:
            print(f'🔐 Agregando {len(new_certs)} certificados nuevos a certifi...')
            with open(certifi_bundle, 'a') as f:
                f.write('\n' + '\n'.join(new_certs))
            print('✅ Certificados del sistema agregados al bundle de certifi')
        else:
            print('✅ Certificados del sistema ya presentes en certifi')
        os.environ['SSL_CERT_FILE']      = certifi_bundle
        os.environ['REQUESTS_CA_BUNDLE'] = certifi_bundle
        os.environ['CURL_CA_BUNDLE']     = certifi_bundle
        print('✅ Variables de entorno SSL configuradas')

    else:
        print('ℹ️  Linux detectado — usando certificados del sistema directamente.')
        system_ca = '/etc/ssl/certs/ca-certificates.crt'
        if os.path.exists(system_ca):
            os.environ['SSL_CERT_FILE']      = system_ca
            os.environ['REQUESTS_CA_BUNDLE'] = system_ca
            print('✅ Usando /etc/ssl/certs/ca-certificates.crt')

setup_ssl()

✅ Certificados corporativos ya presentes en certifi
✅ Variables de entorno SSL configuradas


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ipywidgets', '-q'])
print('✅ Dependencias listas')

✅ Dependencias listas


In [ ]:
try:
    import key_param
    MONGODB_URI    = key_param.mongodb_uri
    VOYAGE_API_KEY = key_param.voyage_api_key
    print('✅ Credenciales cargadas desde key_param.py')
except ImportError:
    MONGODB_URI    = 'tu_mongodb_uri_aqui'
    VOYAGE_API_KEY = 'tu_voyage_api_key_aqui'
    print('⚠️  Define tus credenciales aquí o en key_param.py')

DB_NAME           = 'beaconlm'
PROJECTS_COLL     = 'beaconlm_projects'   # directorio de proyectos
ACTIVE_PROJECT_ID = None                   # se actualiza al activar un proyecto
VS_COLLECTION     = None                   # colección activa del agente
VS_INDEX_NAME     = None                   # índice activo del agente

# ─── System prompt global ─────────────────────────────────────────────────
# Comportamiento base para todos los agentes.
# Cada proyecto puede sobreescribirlo con su propio prompt.
SYSTEM_PROMPT_GLOBAL = (
    'Always respond in the same language the user uses. '
    'You are a helpful AI assistant that answers questions '
    'based exclusively on the documents loaded into your knowledge base. '
    'Use the provided tools to retrieve relevant information before answering. '
    'Think step-by-step. Do not re-run tools unless necessary. '
    'If the answer is not found in the available documents, '
    'say clearly that you do not have that information. '
    'Available tools: {tool_names}.'
)

SYSTEM_PROMPT = SYSTEM_PROMPT_GLOBAL   # se actualiza al activar un proyecto

print(f'📡 MongoDB URI: {MONGODB_URI[:40]}...')
print(f'🧠 System prompt global configurado ({len(SYSTEM_PROMPT_GLOBAL)} chars)')
print('💡 Modifica SYSTEM_PROMPT_GLOBAL para cambiar el comportamiento base.')


In [4]:
from pymongo import MongoClient
import voyageai
import certifi

def get_mongodb_client():
    """Retorna un cliente MongoDB con TLS configurado."""
    return MongoClient(MONGODB_URI, tlsCAFile=certifi.where())

try:
    client = get_mongodb_client()
    client.admin.command('ping')
    print('✅ Conexión a MongoDB Atlas exitosa')
except Exception as e:
    print(f'❌ Error de conexión: {e}')

✅ Conexión a MongoDB Atlas exitosa


---
## 📁 Sección 2 — Gestión de proyectos

Un **proyecto** agrupa documentos relacionados bajo un identificador único.
Cada proyecto tiene su propia colección en MongoDB, su índice vectorial
y opcionalmente un system prompt específico.

| Función | Descripción |
|---|---|
| `create_project(...)` | Crea un nuevo proyecto |
| `list_projects()` | Lista todos los proyectos disponibles |
| `activate_project(id)` | Activa un proyecto para el agente |
| `update_project(id, ...)` | Actualiza metadatos de un proyecto |
| `delete_project(id)` | Elimina un proyecto y sus documentos |

In [ ]:
from datetime import datetime, timezone

def _projects_col():
    """Retorna la colección del directorio de proyectos."""
    return get_mongodb_client()[DB_NAME][PROJECTS_COLL]


def create_project(
    project_id:   str,
    name:         str,
    description:  str  = '',
    system_prompt: str = None,
) -> dict:
    """
    Crea un nuevo proyecto en el directorio.

    Args:
        project_id:    identificador único (ej: 'mongodb_tech')
        name:          nombre legible del proyecto
        description:   descripción del contenido
        system_prompt: prompt específico del proyecto (None = usa el global)

    Returns:
        dict con los metadatos del proyecto creado
    """
    col = _projects_col()
    if col.find_one({'_id': project_id}):
        print(f'⚠️  El proyecto "{project_id}" ya existe. Usa update_project() para modificarlo.')
        return col.find_one({'_id': project_id})

    doc = {
        '_id':           project_id,
        'name':          name,
        'description':   description,
        'vs_collection': f'proj_{project_id}',
        'index_name':    f'idx_{project_id}',
        'system_prompt': system_prompt,   # None → usa SYSTEM_PROMPT_GLOBAL
        'sources':       [],
        'chunk_count':   0,
        'created_at':    datetime.now(timezone.utc).isoformat(),
        'updated_at':    datetime.now(timezone.utc).isoformat(),
    }
    col.insert_one(doc)
    print(f'✅ Proyecto "{project_id}" creado')
    print(f'   Colección: proj_{project_id}')
    print(f'   Índice:    idx_{project_id}')
    print(f'   Prompt:    {"propio" if system_prompt else "global (hereda SYSTEM_PROMPT_GLOBAL)"}')
    return doc


def list_projects() -> list:
    """
    Lista todos los proyectos disponibles con sus metadatos.
    """
    col = _projects_col()
    projects = list(col.find({}))

    if not projects:
        print('📭 No hay proyectos creados. Usa create_project() para crear uno.')
        return []

    active = ACTIVE_PROJECT_ID or '—'
    print(f'📁 Proyectos disponibles en BEEacon LM  (activo: {active})')
    print(f'{"─"*62}')
    for p in projects:
        marker = ' ← activo' if p['_id'] == ACTIVE_PROJECT_ID else ''
        prompt_info = 'prompt propio' if p.get('system_prompt') else 'prompt global'
        print(f'  🐝 {p["_id"]}{marker}')
        print(f'     Nombre:    {p["name"]}')
        print(f'     Desc:      {p.get("description", "")[:60]}')
        print(f'     Colección: {p["vs_collection"]}  ({p.get("chunk_count",0)} chunks)')
        print(f'     Prompt:    {prompt_info}')
        print(f'     Creado:    {p["created_at"][:10]}')
        print()
    return projects


def activate_project(project_id: str) -> dict:
    """
    Activa un proyecto: carga su colección, índice y prompt en el agente.
    Después de activar, vuelve a ejecutar la Sección 4 para aplicar el cambio.

    Args:
        project_id: identificador del proyecto a activar

    Returns:
        dict con los metadatos del proyecto activado
    """
    global ACTIVE_PROJECT_ID, VS_COLLECTION, VS_INDEX_NAME, SYSTEM_PROMPT

    col = _projects_col()
    project = col.find_one({'_id': project_id})

    if not project:
        print(f'❌ Proyecto "{project_id}" no encontrado. Usa list_projects() para ver los disponibles.')
        return None

    ACTIVE_PROJECT_ID = project_id
    VS_COLLECTION     = project['vs_collection']
    VS_INDEX_NAME     = project['index_name']
    SYSTEM_PROMPT     = project.get('system_prompt') or SYSTEM_PROMPT_GLOBAL

    prompt_src = 'propio' if project.get('system_prompt') else 'global'
    print(f'✅ Proyecto "{project_id}" activado')
    print(f'   Nombre:    {project["name"]}')
    print(f'   Colección: {VS_COLLECTION}')
    print(f'   Índice:    {VS_INDEX_NAME}')
    print(f'   Prompt:    {prompt_src}')
    print(f'   Chunks:    {project.get("chunk_count", 0)}')
    print()
    print('💡 Vuelve a ejecutar la Sección 4 para que el agente use este proyecto.')
    return project


def update_project(
    project_id:    str,
    name:          str  = None,
    description:   str  = None,
    system_prompt: str  = None,
    clear_prompt:  bool = False,
) -> None:
    """
    Actualiza los metadatos de un proyecto existente.

    Args:
        project_id:    identificador del proyecto
        name:          nuevo nombre (None = sin cambio)
        description:   nueva descripción (None = sin cambio)
        system_prompt: nuevo prompt (None = sin cambio)
        clear_prompt:  True = elimina el prompt propio y vuelve al global
    """
    col = _projects_col()
    update = {'updated_at': datetime.now(timezone.utc).isoformat()}
    if name:          update['name']          = name
    if description:   update['description']   = description
    if system_prompt: update['system_prompt'] = system_prompt
    if clear_prompt:  update['system_prompt'] = None

    result = col.update_one({'_id': project_id}, {'$set': update})
    if result.matched_count:
        print(f'✅ Proyecto "{project_id}" actualizado')
    else:
        print(f'❌ Proyecto "{project_id}" no encontrado')


def delete_project(project_id: str, confirm: bool = False) -> None:
    """
    Elimina un proyecto y todos sus documentos.

    Args:
        project_id: identificador del proyecto
        confirm:    debe ser True para proceder (resguardo contra borrado accidental)
    """
    if not confirm:
        print(f'⚠️  Para eliminar "{project_id}" y todos sus documentos:')
        print(f'   delete_project("{project_id}", confirm=True)')
        return

    col     = _projects_col()
    project = col.find_one({'_id': project_id})
    if not project:
        print(f'❌ Proyecto "{project_id}" no encontrado')
        return

    client    = get_mongodb_client()
    db        = client[DB_NAME]
    docs_col  = db[project['vs_collection']]
    n_deleted = docs_col.count_documents({})
    docs_col.drop()
    col.delete_one({'_id': project_id})

    print(f'🗑️  Proyecto "{project_id}" eliminado')
    print(f'   Colección {project["vs_collection"]} eliminada ({n_deleted} chunks)')


print('✅ Funciones de gestión de proyectos definidas')
print('   create_project() | list_projects() | activate_project()')
print('   update_project() | delete_project()')


In [ ]:
# ─── EJEMPLOS DE USO — descomenta y adapta según necesites ────────────────

# Listar proyectos existentes
list_projects()

# ── Crear proyectos (solo la primera vez) ─────────────────────────────────
# create_project(
#     project_id  = 'mongodb_tech',
#     name        = 'MongoDB arquitectura e información técnica',
#     description = 'Documentación técnica oficial de MongoDB',
# )

# create_project(
#     project_id  = 'beebank_manuals',
#     name        = 'BEEbank manuales del usuario y del programador',
#     description = 'Manuales internos BEEbank 2026',
#     system_prompt = (
#         'Always respond in the same language the user uses. '
#         'You are a BEEbank technical assistant. '
#         'Answer only from the official BEEbank manuals. '
#         'Available tools: {tool_names}.'
#     )
# )

# create_project(
#     project_id  = 'beespa_catalog',
#     name        = 'BEE SpA información sobre productos y servicios',
#     description = 'Catálogo de productos y servicios para clientes BEE SpA',
#     system_prompt = (
#         'Always respond in the same language the user uses. '
#         'You are a friendly BEE SpA product advisor. '
#         'Help customers find the right product or service. '
#         'Available tools: {tool_names}.'
#     )
# )

# ── Activar un proyecto ────────────────────────────────────────────────────
# activate_project('mongodb_tech')


---
## 📂 Sección 3 — Carga de documentos

Carga fuentes al proyecto activo. Si el proyecto ya tiene documentos, la celda
de punto de control lo detecta y salta la carga automáticamente.

> ⚠️ **Prerrequisito:** Ejecuta `activate_project('tu_id')` en la Sección 2 antes de cargar.
>
> **Formatos soportados:** PDF · DOCX · PPTX · TXT · MD · URL

In [ ]:
import os, re, time, requests

# ─── FUENTES A CARGAR ─────────────────────────────────────────────────────
# Agrega aquí las rutas o URLs de tus documentos.
# Formatos: PDF | DOCX | PPTX | TXT | MD | URL

SOURCES = [
    # 'documentos/manual_usuario.pdf',
    # 'documentos/presentacion.pptx',
    # 'documentos/informe.docx',
    # 'https://ejemplo.com/pagina',
]

CHUNK_SIZE    = 500   # palabras por chunk
CHUNK_OVERLAP = 50    # solapamiento entre chunks

if ACTIVE_PROJECT_ID:
    print(f'✅ Proyecto activo: "{ACTIVE_PROJECT_ID}"')
    print(f'   Colección destino: {VS_COLLECTION}')
    print(f'   Fuentes configuradas: {len(SOURCES)}')
    print(f'   Chunk size: {CHUNK_SIZE} palabras | Overlap: {CHUNK_OVERLAP} palabras')
else:
    print('❌ No hay proyecto activo.')
    print('   Ejecuta activate_project("tu_id") en la Sección 2 primero.')


In [ ]:
def extract_text_from_pdf(path: str) -> str:
    """Extrae texto de un PDF usando pypdf."""
    try:
        from pypdf import PdfReader
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'pypdf', '-q'])
        from pypdf import PdfReader
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            pages.append(f'[Página {i+1}]\n{text.strip()}')
    return '\n\n'.join(pages)


def extract_text_from_docx(path: str) -> str:
    """Extrae texto de un Word (.docx): párrafos y tablas."""
    try:
        from docx import Document
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-docx', '-q'])
        from docx import Document
    doc = Document(path)
    parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                if cell.text.strip():
                    parts.append(cell.text.strip())
    return '\n\n'.join(parts)


def extract_text_from_pptx(path: str) -> str:
    """Extrae texto de un PowerPoint (.pptx): títulos, cuerpos y notas."""
    try:
        from pptx import Presentation
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-pptx', '-q'])
        from pptx import Presentation
    prs = Presentation(path)
    slides = []
    for i, slide in enumerate(prs.slides):
        texts = []
        for shape in slide.shapes:
            if hasattr(shape, 'text') and shape.text.strip():
                texts.append(shape.text.strip())
        if slide.has_notes_slide:
            notes = slide.notes_slide.notes_text_frame.text.strip()
            if notes:
                texts.append(f'[Notas: {notes}]')
        if texts:
            slides.append(f'[Diapositiva {i+1}]\n' + '\n'.join(texts))
    return '\n\n'.join(slides)


def extract_text_from_url(url: str) -> str:
    """Descarga una URL y extrae el texto limpio del HTML."""
    try:
        from bs4 import BeautifulSoup
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'beautifulsoup4', '-q'])
        from bs4 import BeautifulSoup
    headers = {'User-Agent': 'Mozilla/5.0'}
    resp = requests.get(url, headers=headers, timeout=15,
                        verify=os.environ.get('SSL_CERT_FILE', True))
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
        tag.decompose()
    text = soup.get_text(separator='\n')
    return re.sub(r'\n{3,}', '\n\n', text).strip()


def load_source(source: str) -> tuple:
    """
    Carga una fuente y retorna (texto, nombre).
    Formatos soportados: PDF, DOCX, PPTX, TXT, MD y URLs.
    """
    if source.startswith('http://') or source.startswith('https://'):
        print(f'   🌐 Descargando URL: {source[:70]}...')
        return extract_text_from_url(source), source.split('/')[-1] or source[:50]
    ext  = os.path.splitext(source)[1].lower()
    name = os.path.basename(source)
    if ext == '.pdf':
        print(f'   📄 Leyendo PDF: {name}')
        return extract_text_from_pdf(source), name
    elif ext == '.docx':
        print(f'   📝 Leyendo Word: {name}')
        return extract_text_from_docx(source), name
    elif ext == '.pptx':
        print(f'   📊 Leyendo PowerPoint: {name}')
        return extract_text_from_pptx(source), name
    elif ext in ('.txt', '.md'):
        print(f'   📋 Leyendo archivo: {name}')
        with open(source, 'r', encoding='utf-8') as f:
            return f.read(), name
    else:
        raise ValueError(
            f'Formato no soportado: "{ext}". '
            'Usa PDF, DOCX, PPTX, TXT, MD o URL.'
        )


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> list:
    """Divide el texto en chunks con ventana deslizante."""
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        chunk = ' '.join(words[start:start + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


def ensure_vector_index(collection, index_name: str):
    """Crea el índice vectorial si no existe."""
    model = {
        'name': index_name, 'type': 'vectorSearch',
        'definition': {'fields': [{
            'type': 'vector', 'path': 'embedding',
            'numDimensions': 512, 'similarity': 'cosine'
        }]}
    }
    try:
        collection.create_search_index(model=model)
        print(f"   ✅ Índice '{index_name}' creado — espera 1-2 min para que esté activo.")
    except Exception as e:
        if 'already exists' in str(e).lower():
            print(f"   ℹ️  Índice '{index_name}' ya existe.")
        else:
            raise

print('✅ Funciones de extracción y chunking definidas')
print('   PDF | DOCX | PPTX | TXT | MD | URL')


In [ ]:
# ─── PUNTO DE CONTROL ─────────────────────────────────────────────────────
MY_DOCS_LOADED = False

if not ACTIVE_PROJECT_ID:
    print('❌ No hay proyecto activo. Ejecuta activate_project() primero.')
else:
    client   = get_mongodb_client()
    my_col   = client[DB_NAME][VS_COLLECTION]
    existing = my_col.count_documents({})

    print(f'📊 Proyecto: "{ACTIVE_PROJECT_ID}"')
    print(f'   Colección: {VS_COLLECTION}  →  {existing} chunks existentes')

    if existing > 0:
        MY_DOCS_LOADED = True
        print('\n✅ Documentos ya cargados — la carga se saltará automáticamente.')
        print('   Usa delete_project() + create_project() para recargar desde cero.')
    elif not SOURCES:
        print('\n⚠️  SOURCES está vacío — agrega rutas o URLs y vuelve a ejecutar.')
    else:
        print(f'\n⚠️  Colección vacía — se procederá con la carga de {len(SOURCES)} fuente(s).')


In [ ]:
# ─── CARGA, CHUNKING Y EMBEDDINGS ─────────────────────────────────────────
if MY_DOCS_LOADED:
    print('⏭️  Documentos ya cargados — celda omitida.')
elif not ACTIVE_PROJECT_ID:
    print('⏭️  Sin proyecto activo — celda omitida.')
elif not SOURCES:
    print('⏭️  SOURCES vacío — celda omitida.')
else:
    import voyageai as _vai
    vo           = _vai.Client(api_key=VOYAGE_API_KEY)
    total_chunks = 0
    new_sources  = []

    print(f'📥 Cargando {len(SOURCES)} fuente(s) en "{ACTIVE_PROJECT_ID}"...\n')

    for s_idx, source in enumerate(SOURCES):
        print(f'[{s_idx+1}/{len(SOURCES)}] {source}')
        try:
            text, name = load_source(source)
        except Exception as e:
            print(f'   ❌ Error: {e}')
            continue

        words  = text.split()
        chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)
        print(f'   ✂️  {len(words)} palabras  →  {len(chunks)} chunks')

        for i, chunk in enumerate(chunks):
            try:
                embedding = vo.embed(
                    chunk, model='voyage-3-lite', input_type='document'
                ).embeddings[0]
            except Exception as e:
                if 'rate' in str(e).lower():
                    print(f'   ⏳ Rate limit — esperando 60s... (chunk {i+1}/{len(chunks)})')
                    time.sleep(60)
                    embedding = vo.embed(
                        chunk, model='voyage-3-lite', input_type='document'
                    ).embeddings[0]
                else:
                    raise

            my_col.insert_one({
                'project_id':  ACTIVE_PROJECT_ID,
                'source':      source,
                'source_name': name,
                'chunk_index': i,
                'body':        chunk,
                'embedding':   embedding,
            })
            total_chunks += 1
            if (i + 1) % 10 == 0:
                print(f'   ✅ {i+1}/{len(chunks)} chunks insertados')

        new_sources.append(source)
        print(f"   ✅ '{name}' cargado\n")

    # Actualizar metadatos del proyecto
    _projects_col().update_one(
        {'_id': ACTIVE_PROJECT_ID},
        {'$inc':  {'chunk_count': total_chunks},
         '$push': {'sources': {'$each': new_sources}},
         '$set':  {'updated_at': datetime.now(timezone.utc).isoformat()}}
    )

    print(f'🎉 Carga finalizada — {total_chunks} chunks en "{VS_COLLECTION}"')

    # Crear índice vectorial
    print('\n📐 Verificando índice vectorial...')
    ensure_vector_index(my_col, VS_INDEX_NAME)
    MY_DOCS_LOADED = True


---
## 🧠 Sección 4 — Agente RAG

Inicializa el agente con el proyecto activo.
Vuelve a ejecutar esta sección cada vez que cambies de proyecto.

In [10]:
from typing import List, Annotated
from typing_extensions import TypedDict
from langchain.agents import tool
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import ToolMessage
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.mongodb import MongoDBSaver

print('✅ Imports del agente cargados')

✅ Imports del agente cargados


/Users/zpino/dev/mongodb/github/mdb-university/AI-Agents-with-MongoDB/01-Building-AI-Agents-with-MongoDB/.venv/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
from typing import List

def get_vs_collection():
    """Retorna la colección vectorial del proyecto activo."""
    if not VS_COLLECTION:
        raise RuntimeError('No hay proyecto activo. Ejecuta activate_project() primero.')
    return get_mongodb_client()[DB_NAME][VS_COLLECTION]


def generate_embedding(text: str) -> List[float]:
    """Genera embedding para un texto usando Voyage AI."""
    import voyageai as _vai
    vo = _vai.Client(api_key=VOYAGE_API_KEY)
    return vo.embed(text, model='voyage-3-lite', input_type='query').embeddings[0]


print(f'✅ Helpers definidos')
print(f'   Proyecto activo: {ACTIVE_PROJECT_ID or "ninguno"}')
print(f'   Colección activa: {VS_COLLECTION or "ninguna"}')


In [ ]:
@tool
def get_information_for_question_answering(user_query: str) -> str:
    """
    Recupera documentos relevantes del proyecto activo
    usando búsqueda vectorial semántica.

    Args:
        user_query: La pregunta del usuario.

    Returns:
        Los documentos recuperados con su fuente de origen.
    """
    query_embedding = generate_embedding(user_query)
    vs_col = get_vs_collection()

    pipeline = [
        {'$vectorSearch': {
            'index':         VS_INDEX_NAME,
            'path':          'embedding',
            'queryVector':   query_embedding,
            'numCandidates': 150,
            'limit':         5,
        }},
        {'$project': {
            '_id': 0, 'body': 1, 'source_name': 1,
            'score': {'$meta': 'vectorSearchScore'}
        }}
    ]
    results = list(vs_col.aggregate(pipeline))
    if not results:
        return 'No se encontraron documentos relevantes en el proyecto activo.'

    return '\n\n'.join([
        f"[Fuente: {doc.get('source_name','?')}  score: {doc.get('score',0):.3f}]\n{doc.get('body','')}"
        for doc in results
    ])


tools         = [get_information_for_question_answering]
tools_by_name = {t.name: t for t in tools}
print('✅ Herramientas definidas')
print(f'   🔧 {tools[0].name}')


In [ ]:
class GraphState(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: GraphState, llm_with_tools) -> GraphState:
    return {'messages': [llm_with_tools.invoke(state['messages'])]}


def tool_node(state: GraphState) -> GraphState:
    result = []
    for tool_call in state['messages'][-1].tool_calls:
        observation = tools_by_name[tool_call['name']].invoke(tool_call['args'])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call['id']))
    return {'messages': result}


def route_tools(state: GraphState):
    last = state['messages'][-1]
    return 'tools' if (hasattr(last, 'tool_calls') and last.tool_calls) else END


def build_agent(mongodb_client):
    """Construye el grafo del agente con el proyecto y prompt activos."""
    if not ACTIVE_PROJECT_ID:
        print('❌ No hay proyecto activo. Ejecuta activate_project() en la Sección 2.')
        return None

    llm    = ChatOllama(temperature=0, model='qwen2.5:14b')
    prompt = ChatPromptTemplate.from_messages([
        ('system', SYSTEM_PROMPT),
        MessagesPlaceholder(variable_name='messages'),
    ])
    prompt         = prompt.partial(tool_names=', '.join([t.name for t in tools]))
    llm_with_tools = prompt | llm.bind_tools(tools)

    graph = StateGraph(GraphState)
    graph.add_node('agent', lambda state: agent_node(state, llm_with_tools))
    graph.add_node('tools', tool_node)
    graph.add_edge(START, 'agent')
    graph.add_edge('tools', 'agent')
    graph.add_conditional_edges('agent', route_tools, {'tools': 'tools', END: END})

    checkpointer = MongoDBSaver(mongodb_client)
    app_graph    = graph.compile(checkpointer=checkpointer)

    from IPython.display import Image, display
    display(Image(app_graph.get_graph().draw_mermaid_png()))
    return app_graph


mongodb_client = get_mongodb_client()
app = build_agent(mongodb_client)
if app:
    print(f'✅ Agente BEEacon LM listo')
    print(f'   Proyecto:  {ACTIVE_PROJECT_ID}')
    print(f'   Colección: {VS_COLLECTION}')
    print(f'   Prompt:    {"propio" if _projects_col().find_one({"_id":ACTIVE_PROJECT_ID},{"system_prompt":1}).get("system_prompt") else "global"}')


---
## 💬 Sección 5 — Chat interactivo

El agente responde en base al proyecto activo.
Presiona **"Nueva sesión"** para iniciar una conversación desde cero.

In [14]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import uuid

# ─── VARIABLES GLOBALES ───────────────────────────────────────────────────────
THREAD_ID    = 'session-1'
chat_history = []
_processing  = False

# Cerrar widgets previos si existen en el namespace
for _var_name in ['chat_output', 'text_input', 'send_button', 
                  'clear_button', 'status_label', 'input_row', '_chat_html']:
    try:
        globals()[_var_name].close()
    except (NameError, Exception):
        pass

def ask_agent(question: str, thread_id: str) -> str:
    config    = {'configurable': {'thread_id': thread_id}}
    input_msg = {'messages': [('user', question)]}
    final_answer = ''
    for output in app.stream(input_msg, config):
        for key, value in output.items():
            if key == 'agent' and value['messages']:
                last = value['messages'][-1]
                if hasattr(last, 'content') and last.content:
                    final_answer = last.content
    return final_answer

def render_chat():
    css = """
    <style>
        .chat-container { font-family: 'Segoe UI', sans-serif; max-width: 800px;
            padding: 12px; background: #1e1e2e; border-radius: 12px; margin-bottom: 12px; }
        .chat-header { color: #cdd6f4; font-size: 13px; margin-bottom: 12px;
            padding-bottom: 8px; border-bottom: 1px solid #313244; }
        .msg-user { background: #313244; color: #cdd6f4; border-radius: 12px 12px 4px 12px;
            padding: 10px 14px; margin: 6px 0 6px 80px; font-size: 14px; line-height: 1.5; }
        .msg-agent { background: #181825; color: #a6e3a1; border-radius: 12px 12px 12px 4px;
            padding: 10px 14px; margin: 6px 80px 6px 0; font-size: 14px; line-height: 1.6;
            border-left: 3px solid #a6e3a1; }
        .msg-label { font-size: 11px; font-weight: 600; margin-bottom: 4px; opacity: 0.6; }
        .label-user  { color: #89b4fa; text-align: right; }
        .label-agent { color: #a6e3a1; }
        .empty-state { color: #585b70; text-align: center; padding: 24px; font-size: 13px; }
    </style>
    """
    header = f"<div class='chat-container'><div class='chat-header'>🤖 MongoDB RAG Agent &nbsp;|&nbsp; Thread: <code style='color:#89b4fa'>{THREAD_ID}</code></div>"
    if not chat_history:
        body = "<div class='empty-state'>Aún no hay mensajes. ¡Haz una pregunta!</div>"
    else:
        body = ''
        for role, text in chat_history:
            text_safe = text.replace('\n', '<br>')
            if role == 'user':
                body += f"<div class='msg-label label-user'>Tú</div><div class='msg-user'>{text_safe}</div>"
            else:
                body += f"<div class='msg-label label-agent'>🤖 Agente</div><div class='msg-agent'>{text_safe}</div>"
    return css + header + body + '</div>'


# ─── CREAR WIDGETS ────────────────────────────────────────────────────────────
# USAR HTML WIDGET en lugar de Output widget para evitar duplicación
_chat_html = widgets.HTML(value=render_chat())

text_input = widgets.Text(
    placeholder='Escribe tu pregunta sobre MongoDB...',
    continuous_update=False,
    layout=widgets.Layout(width='75%')
)
send_button = widgets.Button(
    description='Enviar',
    button_style='primary',
    layout=widgets.Layout(width='12%')
)
clear_button = widgets.Button(
    description='Nueva sesión',
    button_style='warning',
    layout=widgets.Layout(width='13%')
)
status_label = widgets.Label(value='')
input_row = widgets.HBox([text_input, send_button, clear_button])

# ─── HANDLERS ─────────────────────────────────────────────────────────────────
def refresh_display():
    """Actualiza el widget HTML directamente, sin display() ni clear_output()"""
    _chat_html.value = render_chat()

def on_send(b):
    global THREAD_ID, _processing
    if _processing:
        return
    question = text_input.value.strip()
    if not question:
        return

    _processing = True
    text_input.unobserve(on_enter, names='value')
    text_input.value = ''
    text_input.disabled = True
    send_button.disabled = True
    status_label.value = '⏳ El agente está pensando...'

    chat_history.append(('user', question))
    refresh_display()  # ← Actualiza el HTML directamente

    try:
        answer = ask_agent(question, THREAD_ID)
        chat_history.append(('agent', answer))
    except Exception as e:
        chat_history.append(('agent', f'❌ Error: {str(e)}'))

    status_label.value = ''
    text_input.disabled = False
    _processing = False
    text_input.observe(on_enter, names='value')
    send_button.disabled = False
    refresh_display()  # ← Actualiza el HTML directamente

def on_clear(b):
    global THREAD_ID, chat_history
    THREAD_ID = f'session-{uuid.uuid4().hex[:6]}'
    chat_history = []
    refresh_display()
    status_label.value = f'🆕 Nueva sesión: {THREAD_ID}'

def on_enter(change):
    if change['new'].strip() and not _processing:
        on_send(None)

# ─── CONECTAR EVENTOS ─────────────────────────────────────────────────────────
text_input.observe(on_enter, names='value')
send_button.on_click(on_send)
clear_button.on_click(on_clear)

# ─── MOSTRAR UNA SOLA VEZ ─────────────────────────────────────────────────────
display(_chat_html, input_row, status_label)
print("💬 Chat listo. Escribe tu pregunta y presiona Enter o 'Enviar'.")

HTML(value="\n    <style>\n        .chat-container { font-family: 'Segoe UI', sans-serif; max-width: 800px;\n …

Label(value='')

💬 Chat listo. Escribe tu pregunta y presiona Enter o 'Enviar'.


---
## 🔍 Sección 6 — Diagnóstico *(opcional)*

Celdas utilitarias para inspeccionar el estado del sistema.

In [ ]:
# ─── Estado completo de proyectos y colecciones ───────────────────────────
client = get_mongodb_client()
db     = client[DB_NAME]

print(f'📊 BEEacon LM  —  base de datos: "{DB_NAME}"')
print(f'   Proyecto activo: {ACTIVE_PROJECT_ID or "ninguno"}')
print()

projects = list(_projects_col().find({}))
if not projects:
    print('  Sin proyectos creados todavía.')
else:
    for p in projects:
        col_count = db[p['vs_collection']].count_documents({})
        activo    = ' ← activo' if p['_id'] == ACTIVE_PROJECT_ID else ''
        print(f'  🐝 {p["_id"]}{activo}')
        print(f'     {p["name"]}')
        print(f'     Colección: {p["vs_collection"]}  ({col_count} chunks en BD)')
        print(f'     Índice:    {p["index_name"]}')
        print(f'     Prompt:    {"propio" if p.get("system_prompt") else "global"}')
        print()


In [ ]:
# ─── Probar búsqueda vectorial directamente ───────────────────────────────
if not ACTIVE_PROJECT_ID:
    print('❌ Activa un proyecto primero con activate_project()')
else:
    test_query = 'Escribe aquí tu pregunta de prueba'
    print(f'🔍 Proyecto: "{ACTIVE_PROJECT_ID}"')
    print(f'   Query: "{test_query}"\n')
    result = get_information_for_question_answering.invoke({'user_query': test_query})
    print(result[:800] + '...' if len(result) > 800 else result)


In [ ]:
# ─── Ver historial del chat de la sesión actual ───────────────────────────
print(f"📜 Proyecto: '{ACTIVE_PROJECT_ID}'  |  Thread: '{THREAD_ID}'")
if not chat_history:
    print('   Sin mensajes en esta sesión.')
for i, (role, msg) in enumerate(chat_history):
    icon = '👤' if role == 'user' else '🤖'
    print(f'\n[{i+1}] {icon} {role.upper()}:')
    print(f'    {msg[:200]}{"..." if len(msg) > 200 else ""}')


In [ ]:
# ─── Diagnóstico SSL y conectividad ──────────────────────────────────────────
system_pem = os.path.expanduser('~/Desktop/system-certs.pem')
print(f'1. system-certs.pem existe: {os.path.exists(system_pem)}')

if sys.platform == 'darwin':
    kasp = subprocess.run(['security', 'find-certificate', '-c', 'Kaspersky', '-a', '-p'],
        capture_output=True, text=True)
    kasp_cert = kasp.stdout.strip()
    print(f"2. Cert Kaspersky en Keychain: {'Sí' if kasp_cert else 'No'}")
    if os.path.exists(system_pem):
        with open(system_pem) as f:
            sc = f.read()
        print(f"3. Cert Kaspersky en system-certs.pem: {'Sí' if kasp_cert[:64] in sc else 'NO ❌'}")
    with open(certifi.where()) as f:
        cb = f.read()
    print(f"4. Cert Kaspersky en certifi bundle: {'Sí' if kasp_cert[:64] in cb else 'NO ❌'}")

print(f"5. SSL_CERT_FILE: {os.environ.get('SSL_CERT_FILE', 'NO DEFINIDA ❌')}")

print('\n6. Probando conexión a MongoDB Atlas...')
try:
    c = get_mongodb_client()
    c.admin.command('ping')
    print('   ✅ Conexión OK')
except Exception as e:
    print(f'   ❌ {str(e)[:100]}')

print('\n7. IP pública actual:')
r = subprocess.run(['curl', '-s', 'https://api.ipify.org'], capture_output=True, text=True)
print(f'   {r.stdout}')